In [1]:
!pip install -q transformers accelerate torch librosa soundfile fastapi uvicorn pyngrok python-multipart
!pip install -q "nemo_toolkit[asr]"
!pip install -q groq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

if hf_token:
    print("Hugging Face token connected successfully!")
else:
    print("HF_TOKEN not found. Check Colab Secrets.")

Hugging Face token connected successfully!


In [3]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    repo_id="bodhan-ai/indic-transcribe-flex",
    token=hf_token
)

print("AudioMentor AI Model Downloaded Successfully!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

AudioMentor AI Model Downloaded Successfully!


In [4]:
import sys

sys.path.insert(0, model_dir)

from indic_transcribe import IndicTranscribe

asr = IndicTranscribe.from_pretrained(model_dir)

print("AudioMentor AI Model Loaded Successfully!")

Loading weights:   0%|          | 0/1923 [00:00<?, ?it/s]

AudioMentor AI Model Loaded Successfully!


In [5]:
from google.colab import userdata
from groq import Groq

groq_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=groq_key)

print("Groq AI Connected Successfully!")

Groq AI Connected Successfully!


In [6]:

def generate_summary(transcript, summary_language="Auto Detect"):

    language_instruction = {
        "Auto Detect": """
        Write the summary in the same language and writing style as the transcript.
        """,
        "Tamil": """
        Write the summary in Tamil script.
        """,
        "English": """
        Write the summary in English.
        """,
        "Tanglish": """
        Write the Tamil summary using English letters only.
        Use natural Tanglish spelling.
        Do not use Tamil script.
        """
    }

    prompt = f"""
    You are an accurate summarization assistant.

    Summarize the transcript in 1 or 2 short sentences.

    Important rules:
    1. Preserve the exact meaning of the transcript.
    2. Never change the speaker's intention.
    3. Never add assumptions or invented details.
    4. Do not introduce words like careless, happy,
       sad or other descriptions unless explicitly stated.
    5. Keep the important actions and events.
    6. Correctly understand spoken Tanglish expressions.
    7. Do not translate or change the meaning unnecessarily.

    Summary language:
    {language_instruction[summary_language]}

    Transcript:
    {transcript}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content.strip()


print("AudioMentor AI Summary Function Ready!")


AudioMentor AI Summary Function Ready!


In [7]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="AudioMentor AI")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

print("AudioMentor AI Backend Created!")

AudioMentor AI Backend Created!


In [8]:
@app.post("/process-audio")
async def process_audio(
    file: UploadFile = File(...),
    transcription_language: str = Form("ta"),
    transcription_mode: str = Form("native"),
    summary_language: str = Form("Auto Detect")
):
    import subprocess

    audio_path = "/content/uploaded_audio"
    wav_path = "/content/uploaded_audio_16k.wav"

    with open(audio_path, "wb") as f:
        f.write(await file.read())

    conversion = subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", audio_path,
            "-vn",
            "-ar", "16000",
            "-ac", "1",
            "-c:a", "pcm_s16le",
            wav_path
        ],
        capture_output=True,
        text=True
    )

    if conversion.returncode != 0:
        return {
            "error": "Audio conversion failed",
            "details": conversion.stderr[-2000:]
        }

    transcript = asr.transcribe(
        wav_path,
        lang=transcription_language,
        mode=transcription_mode
    )

    summary = generate_summary(
        transcript,
        summary_language=summary_language
    )

    return {
        "transcript": transcript,
        "summary": summary
    }

print("Audio Processing API Ready!")

Audio Processing API Ready!


In [9]:
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

print("AudioMentor AI Server Started!")

AudioMentor AI Server Started!


In [10]:
from google.colab import userdata
from pyngrok import ngrok

ngrok_token = userdata.get("NGROK_AUTH_TOKEN")

ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(8000).public_url

print("AudioMentor AI Public URL:", public_url)

INFO:     Started server process [762]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


AudioMentor AI Public URL: https://conceal-henna-trodden.ngrok-free.dev


In [11]:
test_transcript = "Naan kaalaiyil seekkiramaaga endhirichu walking ponen"

test_summary = generate_summary(
    test_transcript,
    summary_language="Tanglish"
)

print("Transcript:", test_transcript)
print("AI Summary:", test_summary)

Transcript: Naan kaalaiyil seekkiramaaga endhirichu walking ponen
AI Summary: Naan kaalaiyil seekhiramaaga walking pone.
